# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [26]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania: 3.55s


In [28]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Dziwny przypadek psa nocną porą
2. Koncert oratoryjno-pasyjny AMKP
3. Międzynarodowy Dzień Poezji z Krakowem Miastem Literatury UNESCO
4. Śpiewoterapia
5. Czytanie na dywanie
6. Alicja w Krainie Czarów
7. Impro KRK Underground
8. Jestem obok. Wszyscy w domu
9. Bal
10. Amirova Trio & Iwona Karcz-Wojnarowska: Kiedy tradycja spotyka jazz

Czas wykonania (wątki): 0.77s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [20]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [21]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 12 procesach (rdzeniach)...
Zakończono w czasie 0.35s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [1]:
# Rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
FACTS_COUNT = 20

def fetch_cat_fact():
    """Pobiera pojedynczy fakt o kocie z publicznego API."""
    response = requests.get(CAT_API_URL, timeout=10)
    response.raise_for_status()
    return response.json().get("fact", "Brak faktu")

def run_sequential(n=FACTS_COUNT):
    start = time.perf_counter()
    facts = [fetch_cat_fact() for _ in range(n)]
    elapsed = time.perf_counter() - start
    return facts, elapsed

def run_threaded(n=FACTS_COUNT, workers=10):
    start = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
        facts = list(executor.map(lambda _: fetch_cat_fact(), range(n)))
    elapsed = time.perf_counter() - start
    return facts, elapsed

facts_seq, time_seq = run_sequential()
facts_thr, time_thr = run_threaded()

print(f"Pobrano {len(facts_seq)} faktów sekwencyjnie w {time_seq:.3f}s")
print(f"Pobrano {len(facts_thr)} faktów wielowątkowo w {time_thr:.3f}s")
if time_thr > 0:
    print(f"Przyspieszenie: {time_seq / time_thr:.2f}x")

print("\nPrzykładowe fakty (pierwsze 3):")
for i, fact in enumerate(facts_thr[:3], 1):
    print(f"{i}. {fact}")

Pobrano 20 faktów sekwencyjnie w 16.487s
Pobrano 20 faktów wielowątkowo w 2.434s
Przyspieszenie: 6.77x

Przykładowe fakty (pierwsze 3):
1. The average cat food meal is the equivalent to about five mice.
2. In the original Italian version of Cinderella, the benevolent fairy godmother figure was a cat.
3. According to Hebrew legend, Noah prayed to God for help protecting all the food he stored on the ark from being eaten by rats. In reply, God made the lion sneeze, and out popped a cat.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [2]:
# Rozwiązanie zadania 2
import queue
import threading
import time

TOTAL_NUMBERS = 30
numbers_queue = queue.Queue()
producer_done = threading.Event()
print_lock = threading.Lock()

def producer(limit=TOTAL_NUMBERS):
    for number in range(1, limit + 1):
        numbers_queue.put(number)
        time.sleep(0.01)  # Symulujemy tempo produkcji danych
    producer_done.set()

def consumer(parity_name):
    """Konsument pobiera liczby o wskazanej parzystości z jednej kolejki."""
    remainder = 0 if parity_name == "parzyste" else 1
    processed = []

    while True:
        if producer_done.is_set() and numbers_queue.empty():
            break

        try:
            number = numbers_queue.get(timeout=0.1)
        except queue.Empty:
            continue

        if number % 2 == remainder:
            processed.append(number)
            with print_lock:
                print(f"Konsument {parity_name}: {number}")
            numbers_queue.task_done()
        else:
            numbers_queue.task_done()
            numbers_queue.put(number)
            time.sleep(0.005)

    with print_lock:
        print(f"Konsument {parity_name} zakończył. Przetworzone: {processed}")

producer_thread = threading.Thread(target=producer, name="Producer")
consumer_even_thread = threading.Thread(target=consumer, args=("parzyste",), name="ConsumerEven")
consumer_odd_thread = threading.Thread(target=consumer, args=("nieparzyste",), name="ConsumerOdd")

producer_thread.start()
consumer_even_thread.start()
consumer_odd_thread.start()

producer_thread.join()
numbers_queue.join()
consumer_even_thread.join()
consumer_odd_thread.join()

print("Program zakończył działanie.")

Konsument nieparzyste: 1
Konsument parzyste: 2
Konsument nieparzyste: 3
Konsument parzyste: 4
Konsument nieparzyste: 5
Konsument parzyste: 6
Konsument nieparzyste: 7
Konsument parzyste: 8
Konsument nieparzyste: 9
Konsument parzyste: 10
Konsument nieparzyste: 11
Konsument parzyste: 12
Konsument nieparzyste: 13
Konsument parzyste: 14
Konsument nieparzyste: 15
Konsument parzyste: 16
Konsument nieparzyste: 17
Konsument parzyste: 18
Konsument nieparzyste: 19
Konsument parzyste: 20
Konsument nieparzyste: 21
Konsument parzyste: 22
Konsument nieparzyste: 23
Konsument parzyste: 24
Konsument nieparzyste: 25
Konsument parzyste: 26
Konsument nieparzyste: 27
Konsument parzyste: 28
Konsument nieparzyste: 29
Konsument parzyste: 30
Konsument parzyste zakończył. Przetworzone: [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
Konsument nieparzyste zakończył. Przetworzone: [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29]
Program zakończył działanie.


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [3]:
# Rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def run_parallel_power_sums(start_num=1, end_num=10_000):
    numbers = list(range(start_num, end_num + 1))
    workers = multiprocessing.cpu_count()

    print(f"Uruchamiam obliczenia dla {len(numbers)} liczb na {workers} procesach...")
    start = time.perf_counter()

    with multiprocessing.Pool(processes=workers) as pool:
        results = pool.map(calculate_power_sum, numbers)

    elapsed = time.perf_counter() - start
    print(f"Zakończono w {elapsed:.3f}s")
    print(f"Przykładowe wyniki: {results[:5]}")

if __name__ == "__main__":
    run_parallel_power_sums()

Uruchamiam obliczenia dla 10000 liczb na 16 procesach...
Zakończono w 0.947s
Przykładowe wyniki: [100, 2535301200456458802993406410750, 773066281098016996554691694648431909053161283000, 2142584059011987034055949456454883470029603991710390447068500, 9860761315262647567646607066034827870915080438862787559628486633300780]
